# English → French Translation with Seq2Seq + Pretrained Embeddings (PyTorch)

**Task 1.1 — Modernizing the Machine Translation Model**
 


## 1. Setup and configuration

In [1]:
import os
import re
import io
import math
import time
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import nltk
nltk.download("punkt", quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# pip install rouge-score  (add to requirements)
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [2]:
# ---- Paths to the dataset and to the pretrained vectors ----
DATA_PATH = "eng-fra.txt"            # tab-separated: English<TAB>French  (or a CSV, see loader below)
GLOVE_EN_PATH = "glove.6B.300d.txt"  # English pretrained vectors (GloVe, word2vec-style txt format)
FASTTEXT_FR_PATH = "cc.fr.300.vec"   # French pretrained vectors (FastText .vec format)

WORKSHOP_MODE = False  # True = smaller/faster run for quick testing

CONFIG = {
    "subset_size": 40_000 if WORKSHOP_MODE else None,
    "max_vocab_size": 8_000 if WORKSHOP_MODE else 10_000,
    "max_seq_len": 20 if WORKSHOP_MODE else 40,
    "embed_dim": 300,            # MUST match the pretrained vectors' dimension
    "hidden_dim": 256,           # encoder hidden size (per direction)
    "decoder_hidden": 512,       # 256 * 2 after merging bidirectional states
    "batch_size": 64,
    "epochs": 8 if WORKSHOP_MODE else 15,
    "learning_rate": 1e-3,
    "dropout": 0.2,
    "patience": 3,
    "grad_clip": 1.0,
    "freeze_embeddings": False,  # False = fine-tune pretrained vectors during training
    "num_workers": 0,
}

print("Training config")
for k, v in CONFIG.items():
    print(f"  {k:18s}: {v}")


Training config
  subset_size       : None
  max_vocab_size    : 10000
  max_seq_len       : 40
  embed_dim         : 300
  hidden_dim        : 256
  decoder_hidden    : 512
  batch_size        : 64
  epochs            : 15
  learning_rate     : 0.001
  dropout           : 0.2
  patience          : 3
  grad_clip         : 1.0
  freeze_embeddings : False
  num_workers       : 0


## 2. Load the dataset

Same loader as the workshop notebook: accepts either a tab-separated `eng-fra.txt` (PyTorch tutorial format) or a CSV with English/French columns (Kaggle format).

In [3]:
def load_pairs(path: str) -> pd.DataFrame:
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        cols = {c.lower().strip(): c for c in df.columns}
        eng_col = cols.get("english") or cols.get("en") or list(df.columns)[0]
        fr_col = cols.get("french") or cols.get("fr") or list(df.columns)[1]
        out = df[[eng_col, fr_col]].copy()
        out.columns = ["English", "French"]
        return out.dropna().reset_index(drop=True)
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                rows.append((parts[0], parts[1]))
    return pd.DataFrame(rows, columns=["English", "French"])


df = load_pairs(DATA_PATH)
if CONFIG["subset_size"]:
    df = df.sample(n=min(CONFIG["subset_size"], len(df)), random_state=SEED).reset_index(drop=True)

print(f"Pairs: {len(df):,}")
df.sample(5, random_state=SEED)


Pairs: 135,842


,English,French
131140,"The wind was so strong, we were nearly blown o...",Le vent était tellement fort que nous avons pr...
131601,You can never be happy if you feel envious of ...,Tu ne peux jamais être heureux si tu te sens e...
70924,Let's reconsider the problem.,Reconsidérons le problème !
10276,"Stop it, please.","Cessez, je vous prie !"
10999,You are the one.,Vous êtes celui-là.


## 3. Text cleaning

Same cleaning rules as the workshop: lowercase, expand a few contractions, strip punctuation/digits, collapse spaces.

In [4]:
EN_CONTRACTIONS = {
    "i'm": "i am", "you're": "you are", "he's": "he is", "she's": "she is",
    "it's": "it is", "we're": "we are", "they're": "they are",
    "i've": "i have", "you've": "you have", "we've": "we have", "they've": "they have",
    "i'll": "i will", "you'll": "you will", "he'll": "he will", "she'll": "she will",
    "we'll": "we will", "they'll": "they will",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "can't": "cannot", "couldn't": "could not", "won't": "will not",
    "wouldn't": "would not", "shouldn't": "should not", "haven't": "have not",
    "hasn't": "has not", "hadn't": "had not", "let's": "let us",
}
FR_CONTRACTIONS = {
    "c'est": "ce est", "j'ai": "je ai", "n'est": "ne est",
    "qu'est": "que est", "d'accord": "de accord", "l'est": "le est",
}
FR_KEEP = r"a-zàâäçéèêëîïôùûüœÿ\s"


def _expand(text, table):
    for src_tok, dst_tok in table.items():
        text = re.sub(rf"\b{re.escape(src_tok)}\b", dst_tok, text)
    return text


def clean_english(text):
    text = str(text).lower().strip()
    text = _expand(text, EN_CONTRACTIONS)
    text = re.sub(r"[^a-z\s]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_french(text):
    text = str(text).lower().strip()
    text = _expand(text, FR_CONTRACTIONS)
    text = re.sub(rf"[^{FR_KEEP}]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


df["English"] = df["English"].apply(clean_english)
df["French"] = df["French"].apply(clean_french)
df = df[(df["English"].str.len() > 0) & (df["French"].str.len() > 0)].reset_index(drop=True)
df.sample(5, random_state=SEED)


,English,French
131140,the wind was so strong we were nearly blown of...,le vent était tellement fort que nous avons pr...
131601,you can never be happy if you feel envious of ...,tu ne peux jamais être heureux si tu te sens e...
70924,let us reconsider the problem,reconsidérons le problème
10276,stop it please,cessez je vous prie
10999,you are the one,vous êtes celui là


## 4. Train / validation / test split

10% test, then 10% of the remainder for validation. Vocabulary is built from the training set only.

In [5]:
train_df, test_df = train_test_split(df, test_size=0.10, random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.10, random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,}")
print(f"Valid: {len(val_df):,}")
print(f"Test : {len(test_df):,}")


Train: 110,031
Valid: 12,226
Test : 13,585


## 5. Vocabulary

In [6]:
class Vocabulary:
    PAD, UNK, SOS, EOS = "<pad>", "<unk>", "<start>", "<end>"

    def __init__(self, max_size=10_000):
        self.max_size = max_size
        self.word2idx = {self.PAD: 0, self.UNK: 1, self.SOS: 2, self.EOS: 3}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        self.pad_idx, self.unk_idx = 0, 1
        self.sos_idx, self.eos_idx = 2, 3

    def build(self, sentences):
        counts = Counter()
        for s in sentences:
            counts.update(s.split())
        for word, _ in counts.most_common(self.max_size - 4):
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        return self

    def __len__(self):
        return len(self.word2idx)

    def encode(self, sentence, max_len, add_special=False):
        tokens = sentence.split()
        if add_special:
            tokens = [self.SOS] + tokens + [self.EOS]
        ids = [self.word2idx.get(t, self.unk_idx) for t in tokens][:max_len]
        length = len(ids)
        ids = ids + [self.pad_idx] * (max_len - length)
        return ids, length

    def decode(self, ids, skip_special=True):
        special = {self.PAD, self.UNK, self.SOS, self.EOS} if skip_special else set()
        words = [self.idx2word.get(int(i), self.UNK) for i in ids]
        words = [w for w in words if w not in special]
        return " ".join(words)


src_vocab = Vocabulary(CONFIG["max_vocab_size"]).build(train_df["English"])
tgt_vocab = Vocabulary(CONFIG["max_vocab_size"]).build(train_df["French"])
print(f"English vocab: {len(src_vocab):,}")
print(f"French  vocab: {len(tgt_vocab):,}")


English vocab: 10,000
French  vocab: 10,000


## 6. Pretrained word embeddings (the actual "modernization")

This replaces the workshop's frequency-based / from-scratch embedding with **pretrained vectors**:

- `load_pretrained_vectors` reads a word2vec-style text file (one line per word: `word v1 v2 ... vn`). This format covers **GloVe**, **FastText `.vec`**, and **Word2Vec text export** — so swapping the file swaps the embedding method.
- `build_embedding_matrix` looks up every word in our vocabulary; words not found in the pretrained file get a small random vector instead (so the model can still learn something for them).
- The matrix is passed into `nn.Embedding.from_pretrained(..., freeze=CONFIG["freeze_embeddings"])`, so training **fine-tunes** the vectors rather than freezing them (usually gives the best results for a small in-domain corpus).

In [7]:
def load_pretrained_vectors(path: str, dim: int, wanted_words: set):
    """Read only the lines whose word is in `wanted_words` (keeps memory low)."""
    vectors = {}
    if not path or not os.path.exists(path):
        print(f"[warning] pretrained file not found: {path} -> that side will use random init")
        return vectors
    with io.open(path, "r", encoding="utf-8", newline="\n", errors="ignore") as f:
        first = f.readline().split()
        # fastText .vec files start with a "n_words dim" header line; GloVe files do not.
        if len(first) != 2:
            f.seek(0)
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            if word in wanted_words:
                vec = np.asarray(parts[1:1 + dim], dtype="float32")
                if vec.shape[0] == dim:
                    vectors[word] = vec
    return vectors


def build_embedding_matrix(vocab: Vocabulary, pretrained_path: str, dim: int, name: str):
    matrix = np.random.normal(scale=0.1, size=(len(vocab), dim)).astype("float32")
    matrix[vocab.pad_idx] = np.zeros(dim, dtype="float32")

    found = load_pretrained_vectors(pretrained_path, dim, set(vocab.word2idx.keys()))
    hits = 0
    for word, idx in vocab.word2idx.items():
        if word in found:
            matrix[idx] = found[word]
            hits += 1
    coverage = hits / len(vocab) * 100
    print(f"{name}: {hits}/{len(vocab)} words found in pretrained vectors ({coverage:.1f}% coverage)")
    return torch.tensor(matrix)


src_embed_matrix = build_embedding_matrix(src_vocab, GLOVE_EN_PATH, CONFIG["embed_dim"], "English (GloVe)")
tgt_embed_matrix = build_embedding_matrix(tgt_vocab, FASTTEXT_FR_PATH, CONFIG["embed_dim"], "French (FastText)")


[warning] pretrained file not found: glove.6B.300d.txt -> that side will use random init
English (GloVe): 0/10000 words found in pretrained vectors (0.0% coverage)
[warning] pretrained file not found: cc.fr.300.vec -> that side will use random init
French (FastText): 0/10000 words found in pretrained vectors (0.0% coverage)


## 7. Dataset and DataLoader

In [8]:
class TranslationDataset(Dataset):
    def __init__(self, frame, src_vocab, tgt_vocab, max_len):
        self.src_texts = frame["English"].tolist()
        self.tgt_texts = frame["French"].tolist()
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src_ids, src_len = self.src_vocab.encode(self.src_texts[idx], self.max_len, add_special=False)
        tgt_ids, _ = self.tgt_vocab.encode(self.tgt_texts[idx], self.max_len, add_special=True)
        tgt_in = tgt_ids[:-1]
        tgt_out = tgt_ids[1:]
        return {
            "src": torch.tensor(src_ids, dtype=torch.long),
            "src_len": torch.tensor(src_len, dtype=torch.long),
            "tgt_in": torch.tensor(tgt_in, dtype=torch.long),
            "tgt_out": torch.tensor(tgt_out, dtype=torch.long),
            "src_text": self.src_texts[idx],
            "tgt_text": self.tgt_texts[idx],
        }


def make_loader(frame, shuffle):
    dataset = TranslationDataset(frame, src_vocab, tgt_vocab, CONFIG["max_seq_len"])
    return DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=shuffle,
                       num_workers=CONFIG["num_workers"], drop_last=False)


train_loader = make_loader(train_df, shuffle=True)
val_loader = make_loader(val_df, shuffle=False)
test_loader = make_loader(test_df, shuffle=False)
print("Loaders ready.")


Loaders ready.


## 8. Model: Seq2Seq with attention (architecture unchanged)

Same as the workshop: **BiLSTM encoder → Luong dot-product attention → LSTM decoder**.
The only change vs. the workshop is inside `Encoder`/`Decoder.__init__`: the embedding layer is now built with `nn.Embedding.from_pretrained(...)` instead of a randomly-initialized `nn.Embedding`.

In [9]:
class Encoder(nn.Module):
    def __init__(self, embed_matrix, hidden_dim, pad_idx, dropout=0.2, freeze=False):
        super().__init__()
        vocab_size, embed_dim = embed_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=freeze, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = torch.cat([hidden[0], hidden[1]], dim=-1).unsqueeze(0)
        cell = torch.cat([cell[0], cell[1]], dim=-1).unsqueeze(0)
        return outputs, hidden, cell


class LuongAttention(nn.Module):
    def forward(self, decoder_outputs, encoder_outputs, src_mask=None):
        scores = torch.bmm(decoder_outputs, encoder_outputs.transpose(1, 2))  # (B,T,S)
        if src_mask is not None:
            scores = scores.masked_fill(~src_mask.unsqueeze(1), float("-inf"))
        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, encoder_outputs)  # (B,T,H)
        return context, weights


class Decoder(nn.Module):
    def __init__(self, embed_matrix, hidden_dim, pad_idx, dropout=0.2, freeze=False):
        super().__init__()
        vocab_size, embed_dim = embed_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=freeze, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, batch_first=True)
        self.attention = LuongAttention()
        self.combine = nn.Linear(hidden_dim * 2, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt_in, hidden, cell, encoder_outputs, src_mask):
        embedded = self.dropout(self.embedding(tgt_in))
        dec_outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        context, attn = self.attention(dec_outputs, encoder_outputs, src_mask)
        combined = torch.tanh(self.combine(torch.cat([dec_outputs, context], dim=-1)))
        logits = self.out(combined)
        return logits, hidden, cell, attn


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, pad_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx

    def create_src_mask(self, src):
        return src != self.pad_idx

    def forward(self, src, tgt_in):
        encoder_outputs, hidden, cell = self.encoder(src)
        src_mask = self.create_src_mask(src)
        logits, _, _, _ = self.decoder(tgt_in, hidden, cell, encoder_outputs, src_mask)
        return logits, None


encoder = Encoder(src_embed_matrix, CONFIG["hidden_dim"], src_vocab.pad_idx,
                   CONFIG["dropout"], freeze=CONFIG["freeze_embeddings"])
decoder = Decoder(tgt_embed_matrix, CONFIG["decoder_hidden"], tgt_vocab.pad_idx,
                   CONFIG["dropout"], freeze=CONFIG["freeze_embeddings"])
model = Seq2Seq(encoder, decoder, pad_idx=src_vocab.pad_idx).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")


Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(10000, 300, padding_idx=0)
    (lstm): LSTM(300, 256, batch_first=True, bidirectional=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(10000, 300, padding_idx=0)
    (lstm): LSTM(300, 512, batch_first=True)
    (attention): LuongAttention()
    (combine): Linear(in_features=1024, out_features=512, bias=True)
    (out): Linear(in_features=512, out_features=10000, bias=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
)

Trainable parameters: 14,464,656


## 9. Loss, optimizer, scheduler

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)


def token_accuracy(logits, targets, pad_idx):
    preds = logits.argmax(dim=-1)
    mask = targets != pad_idx
    if mask.sum() == 0:
        return 0.0
    return (preds[mask] == targets[mask]).float().mean().item()


def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = total_acc = 0.0
    n_batches = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for batch in loader:
            src = batch["src"].to(device)
            tgt_in = batch["tgt_in"].to(device)
            tgt_out = batch["tgt_out"].to(device)

            logits, _ = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                optimizer.step()

            total_loss += loss.item()
            total_acc += token_accuracy(logits, tgt_out, tgt_vocab.pad_idx)
            n_batches += 1
    return total_loss / max(n_batches, 1), total_acc / max(n_batches, 1)


## 10. Training loop (early stopping + checkpointing)

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val = math.inf
best_state = None
stale = 0
t0 = time.time()

header = "epoch", "train_loss", "val_loss", "train_acc", "val_acc", "time"
print(f"{header[0]:>6}  {header[1]:>11}  {header[2]:>9}  {header[3]:>10}  {header[4]:>8}  {header[5]:>6}")
print("-" * 64)

for epoch in range(1, CONFIG["epochs"] + 1):
    epoch_start = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - epoch_start
    print(f"{epoch:6d}  {train_loss:11.4f}  {val_loss:9.4f}  {train_acc:10.3f}  {val_acc:8.3f}  {elapsed:5.0f}s")

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= CONFIG["patience"]:
            print(f"Early stopping at epoch {epoch} (best val loss = {best_val:.4f})")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

print(f"\nFinished in {(time.time() - t0) / 60:.1f} min. Best val loss: {best_val:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs_x = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs_x, history["train_loss"], marker="o", label="train")
axes[0].plot(epochs_x, history["val_loss"], marker="o", label="valid")
axes[0].set_title("Cross-entropy loss (pad ignored)"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs_x, history["train_acc"], marker="o", label="train")
axes[1].plot(epochs_x, history["val_acc"], marker="o", label="valid")
axes[1].set_title("Token accuracy (pad ignored)"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 11. Inference: greedy decoding

In [ ]:
@torch.no_grad()
def translate_sentence(sentence: str, max_len: int = None, return_attention: bool = False):
    model.eval()
    max_len = max_len or CONFIG["max_seq_len"]
    cleaned = clean_english(sentence)
    src_ids, src_len = src_vocab.encode(cleaned, CONFIG["max_seq_len"], add_special=False)
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)

    encoder_outputs, hidden, cell = model.encoder(src)
    src_mask = model.create_src_mask(src)

    token = torch.tensor([[tgt_vocab.sos_idx]], device=device)
    output_ids, attn_steps = [], []

    for _ in range(max_len):
        logits, hidden, cell, attn = model.decoder(token, hidden, cell, encoder_outputs, src_mask)
        next_id = int(logits[0, -1].argmax())
        attn_steps.append(attn[0, 0, :src_len].detach().cpu())
        if next_id == tgt_vocab.eos_idx:
            break
        output_ids.append(next_id)
        token = torch.tensor([[next_id]], device=device)

    text = tgt_vocab.decode(output_ids)
    if return_attention:
        attn_mat = torch.stack(attn_steps, dim=0) if attn_steps else torch.zeros(1, src_len)
        return text, cleaned.split()[:src_len], text.split(), attn_mat
    return text


## 12. Evaluation: BLEU and ROUGE

This is the part the task explicitly asks for. Token accuracy (above) is only a training-time sanity check; **BLEU** and **ROUGE-L** are the metrics that actually measure translation quality against the reference French sentences.

- **BLEU**: precision-based n-gram overlap between the generated sentence and the reference — the standard MT metric.
- **ROUGE-L**: longest-common-subsequence overlap, borrowed from summarization but commonly reported alongside BLEU for a fuller picture (recall-oriented, more forgiving of word order).

In [ ]:
def evaluate_bleu_rouge(frame, n_samples=None, seed=SEED):
    rows = frame if n_samples is None else frame.sample(n=min(n_samples, len(frame)), random_state=seed)

    references, hypotheses = [], []
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
    rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for _, row in rows.iterrows():
        pred = translate_sentence(row["English"])
        ref = row["French"]

        hypotheses.append(pred.split())
        references.append([ref.split()])  # corpus_bleu expects list-of-references per sentence

        scores = rouge.score(ref, pred)
        for key in rouge_scores:
            rouge_scores[key].append(scores[key].fmeasure)

    smoothie = SmoothingFunction().method4
    bleu = corpus_bleu(references, hypotheses, smoothing_function=smoothie)

    results = {"BLEU": bleu}
    for key, vals in rouge_scores.items():
        results[f"{key}_F1"] = float(np.mean(vals))
    return results


# Full test set can be slow on CPU; set n_samples=None to use all of test_df
metrics = evaluate_bleu_rouge(test_df, n_samples=500)
print("Test-set translation quality:")
for k, v in metrics.items():
    print(f"  {k:10s}: {v:.4f}")


## 13. Qualitative results

In [ ]:
def show_translations(n=10, seed=7):
    rng = random.Random(seed)
    indices = rng.sample(range(len(test_df)), k=min(n, len(test_df)))
    rows = []
    for i in indices:
        eng = test_df.loc[i, "English"]
        ref = test_df.loc[i, "French"]
        pred = translate_sentence(eng)
        rows.append({"English": eng, "Reference (FR)": ref, "Predicted (FR)": pred})
    return pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 80)
show_translations(n=12, seed=7)


## 14. Save the model

In [ ]:
ckpt_path = Path("seq2seq_en_fr_glove.pt")
torch.save({
    "model_state": model.state_dict(),
    "config": CONFIG,
    "src_word2idx": src_vocab.word2idx,
    "tgt_word2idx": tgt_vocab.word2idx,
    "test_metrics": metrics,
}, ckpt_path)
print(f"Saved checkpoint -> {ckpt_path.resolve()}")
